In [2]:
import pandas
from rdkit import Chem
from rdkit.Chem import Descriptors

df = pandas.read_csv('skin_irritation_cleaned.csv')
df.shape

df.columns

desc_total = []
for index, row in df.iterrows():
    smi = row['SMILES']
    mol = Chem.MolFromSmiles(smi)
    desc = Descriptors.CalcMolDescriptors(mol)
    desc_total.append(desc)

df_desc = pandas.DataFrame(desc_total)

df_desc

df_id = df[['Chemical_Name','SMILES','label']]

df_final = pandas.concat([df_id,df_desc],axis=1) #axis를 0으로 하면 어떻게 결과가 바뀔까?

df_final #2D descriptor를 csv file로 저장

import numpy as np
from rdkit.Chem import AllChem

fpgen = AllChem.GetRDKitFPGenerator()

fp_total = []
for index, row in df.iterrows():
    smi = row['SMILES']
    mol = Chem.MolFromSmiles(smi)
    fp = fpgen.GetFingerprint(mol)
    fp_array = np.zeros((0,), dtype=int)
    Chem.DataStructs.ConvertToNumpyArray(fp, fp_array)
    fp_total.append(fp_array)

df_fp = pd.DataFrame(fp_total, columns=[f'bit_{i}' for i in range(len(fp_total[0]))])

df_fp #Chemical_Name, SMILES, label 컬럼 추가한 뒤 별도 csv 파일로 저장



# smiles 코드가 단일 구조가 아니라면 '.'이 포함되어 있음.
# 단일 구조가 아닌 물질들만 리스트를 확인하고, 제거해야 될 것과 제거하면 안되는 물질 구분해보기.
# 현재 대부분이 유기 화합물이기 때문에, 유기화합물이 염(salt)가 추가된 경우는, 염을 제거한 smiles 코드를 남겨야함.
# 무기화합물이라면 제거. 유효 성분이 몇개인지 확인이 어려운 경우도 제거.

#https://www.rdkit.org/docs/GettingStartedInPython.html#list-of-available-descriptors
#fingerprint 종류는 매우 다양, 종류별로 모두 계산해서 저장해보기


'''
descriptor 추가로 계산하는 방법 (https://github.com/marcabrus/VegaDescriptor-Calculation)
모델 성능을 올리기 위해 다양한 descriptor 계산 필요.

1. Descriptors vega.zip 다운받기 ( <> Code -> Download ZIP )
2. 실행 파일: prompt
(sublime으로 열어서 명령어 살펴보기, 명령어 설명은 spiegazione prompt.txt에 기재되어 있음.)
3. input 예시 (molecules.txt)
4. output 예시 (out.txt)
'''



'\ndescriptor 추가로 계산하는 방법 (https://github.com/marcabrus/VegaDescriptor-Calculation)\n모델 성능을 올리기 위해 다양한 descriptor 계산 필요.\n\n1. Descriptors vega.zip 다운받기 ( <> Code -> Download ZIP )\n2. 실행 파일: prompt\n(sublime으로 열어서 명령어 살펴보기, 명령어 설명은 spiegazione prompt.txt에 기재되어 있음.)\n3. input 예시 (molecules.txt)\n4. output 예시 (out.txt)\n'

In [45]:
df_filtered = df_filtered.drop_duplicates(subset='SMILES')

In [47]:
print("중복 제거 전:", df_filtered.shape[0])

df_filtered = df_filtered.drop_duplicates(subset='SMILES')

print("중복 제거 후:", df_filtered.shape[0])

중복 제거 전: 48
중복 제거 후: 48


In [7]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover

# 1. 데이터 불러오기
df = pd.read_csv('skin_irritation_cleaned.csv')

# 2. | 포함된 구조 제거
df_clean = df.loc[~df['SMILES'].str.contains('|', regex=False, na=False)].copy()

# 3. salt remover 준비
remover = SaltRemover()

# 4. salt 제거 + 가장 큰 분자만 남기기
clean_smi_list = []

for idx, row in df_clean.iterrows():
    smi = row['SMILES']
    
    if pd.isna(smi) or str(smi).strip() == '':
        clean_smi_list.append('')
        continue
    
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        clean_smi_list.append('')
        continue
    
    if '.' in smi:
        stripped_mol = remover.StripMol(mol)
        
        if stripped_mol is None:
            clean_smi_list.append('')
            continue
        
        stripped_smi = Chem.MolToSmiles(stripped_mol)
        smi_list = stripped_smi.split('.')
        
        smi_active = ''
        for each_smi in smi_list:
            if len(each_smi) > len(smi_active):
                smi_active = each_smi
        
        clean_smi_list.append(smi_active)
    else:
        clean_smi_list.append(smi)

df_clean['smi_clean'] = clean_smi_list

# 5. standardized smiles 만들기
standardized_list = []

for smi in df_clean['smi_clean']:
    if pd.isna(smi) or str(smi).strip() == '':
        standardized_list.append('')
        continue
    
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        standardized_list.append('')
        continue
    
    standardized_list.append(Chem.MolToSmiles(mol))

df_clean['standardized_smi'] = standardized_list

# 6. 빈 smiles 제거
df_clean = df_clean[df_clean['standardized_smi'] != ''].copy()

# 7. label 충돌 제거
# 같은 standardized_smi인데 label이 2개 이상이면 제외
df_filtered = df_clean.groupby('standardized_smi').filter(
    lambda x: x['label'].nunique() == 1
)

# 8. standardized_smi 기준 중복 제거
df_unique = df_filtered.drop_duplicates(subset='standardized_smi').reset_index(drop=True)

# 9. 최종 개수 확인
print("최종 개수:", df_unique.shape[0])
print(df_unique[['Chemical_Name', 'SMILES', 'standardized_smi', 'label']].head())

최종 개수: 39
               Chemical_Name                                   SMILES  \
0                   Heptanal                                CCCCCCC=O   
1  4-Methylthio benzaldehyde                       CSC1=CC=C(C=O)C=C1   
2            Heptyl butyrate                         CCCCCCCOC(=O)CCC   
3         Hydroxycitronellal  COC(=O)C1=C(C=CC=C1)N=CCC(C)CCCC(C)(C)O   
4            Methyl caproate                             CCCCCC(=O)OC   

                     standardized_smi  label  
0                           CCCCCCC=O      1  
1                     CSc1ccc(C=O)cc1      0  
2                    CCCCCCCOC(=O)CCC      0  
3  COC(=O)c1ccccc1N=CCC(C)CCCC(C)(C)O      0  
4                        CCCCCC(=O)OC      0  


In [8]:
df_vega = df_unique[['Chemical_Name', 'standardized_smi', 'label']].copy()
df_vega = df_vega.reset_index(drop=True)

print("VEGA 입력용 개수:", df_vega.shape[0])

df_vega['standardized_smi'].to_csv('molecules.txt', index=False, header=False)

print("molecules.txt 저장 완료")

VEGA 입력용 개수: 39
molecules.txt 저장 완료
